In [0]:
from pyspark.sql.functions import col, lit, current_timestamp

# --- SILVER LAYER: Cleaning & Transformation ---

# 1. Read from the Bronze table
bronze_data = spark.table("main.market_project.bronze_bank_prices")

# 2. Standardize data
silver_df = bronze_data.select(
    col("Datetime").cast("timestamp").alias("event_timestamp"),
    col("Entity").cast("string").alias("bank_name"),
    col("Open").cast("double").alias("open_price"),
    col("High").cast("double").alias("high_price"),
    col("Low").cast("double").alias("low_price"),
    col("Close").cast("double").alias("close_price"),
    col("Volume").cast("long").alias("volume"),
    current_timestamp().alias("processing_timestamp")
).filter(col("event_timestamp").isNotNull())

# 4. Save to Silver table
target_table = "main.market_project.silver_bank_prices"
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

print(f"Success: Silver table '{target_table}' created without casting errors.")
display(silver_df.limit(10))